<a href="https://colab.research.google.com/github/CefasRepRes/shipwreck-oil-detection-colab/blob/main/CefasCobblesTools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def install_requirements():

  global np, seg, sam, predictor, unet, si, torchvision, torch, SAM2ImagePredictor, plt, build_sam2, pd, pd, cv2, measure, reload, trange, tqdm, pickle, Path, predictor, sam, device, sam_checkpoint
  from google.colab import output
  output.enable_custom_widget_manager()
  %matplotlib ipympl
  import torch
  import torchvision
  print("PyTorch version:", torch.__version__)
  print("Torchvision version:", torchvision.__version__)
  print("CUDA is available:", torch.cuda.is_available())
  import sys
  !{sys.executable} -m pip install opencv-python matplotlib
  !{sys.executable} -m pip install sam2
  !{sys.executable} -m pip install segmenteverygrain
  !{sys.executable} -m pip install rtree
  !wget https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
  from sam2.build_sam import build_sam2
  from sam2.sam2_image_predictor import SAM2ImagePredictor
  sam_checkpoint = "sam2.1_hiera_large.pt"
  device = "cuda"
  sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device)
  predictor = SAM2ImagePredictor(sam)
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import cv2
  from skimage import measure
  from importlib import reload
  import segmenteverygrain as seg
  import segmenteverygrain.interactions as si
  from tqdm import trange, tqdm
  import pickle
  from pathlib import Path
  !wget "https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt"
  # U-Net model (PyTorch)
  unet = seg.load_unet("seg_model_smooth_labels.pt")
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG
  !wget -O WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG
  !wget -O Pixel_calculations_subset.csv https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/Pixel_calculations_subset.csv

Code to create the image dropdown box

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
def Select_image():
  global IMGdropdown, min_area_px, dbs__max_dist, Roundness_Thresh
  # Folder containing the images
  image_dir = Path("/content")

  # Find JPG files, ignoring case
  image_files = sorted([
     f for f in image_dir.iterdir()
      if f.is_file() and f.suffix.lower() == ".jpg"
  ])

  # Dropdown shows filename without extension
  IMGdropdown = widgets.Dropdown(
     options=[f.stem for f in image_files],
     description="Select image:"
  )
  # Slider for pixel filter
  min_area_px = widgets.IntSlider(value=250, min=50, max=750, step=10, description='Min pixel area', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})
  dbs__max_dist = widgets.IntSlider(value=30, min=10, max=500, step=5, description='Prompt distance', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})
  Roundness_Thresh = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1, description='Roundness_threshold', readout=True, readout_format='.1f', orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})

  return display(IMGdropdown, min_area_px,dbs__max_dist, Roundness_Thresh)


In [ ]:
def Run_segmentation():
  global all_grains, labels, mask_all, grain_data, fig, ax, image
  # replace this with the path to your image:
  fname = f'{IMGdropdown.value}.JPG'
  image = np.array(seg.load_img(fname))
  image_pred = seg.predict_image(image, unet, I=256)

  # decreasing the 'dbs_max_dist' parameter results in more SAM prompts
  # (and longer processing times):
  labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=dbs__max_dist.value)

  # SAM segmentation, using the point prompts from the Unet:
  all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
    sam,
    image,
    image_pred,
    coords,
    labels,
    min_area=min_area_px.value,
    plot_image=True,
    remove_edge_grains=False,
    remove_large_objects=False,
  )

In [ ]:
def save_segmented_outputs():
  global grains
  # Convert polygons to Grain objects
  grains = si.polygons_to_grains(all_grains, image=image)
  for g in tqdm(grains, desc='Measuring grains'):
    g.measure()

  pkl_filename = Path(f'{IMGdropdown.value}.JPG').with_suffix(".pkl")
  with open(pkl_filename, "wb") as f:
    pickle.dump((
            all_grains,
            labels,
            mask_all,
            grain_data,),f)
  print(f"Saved segmentation to: {pkl_filename}")

In [ ]:
def grain_size_distribution():
    global pixel_x_m

    # Load pixel calculation table
    pixel_data = pd.read_csv("/content/Pixel_calculations_subset.csv")
    # Filename selected in dropdown
    image_name = f"{IMGdropdown.value}.JPG"

    # Find matching image
    match = pixel_data.loc[
        pixel_data["image"].str.lower() == image_name.lower(),
        "pixel_x_cm"
    ]

    if match.empty:
        raise ValueError(
            f"No pixel_x_cm value found for {image_name}"
        )

    if len(match) > 1:
        raise ValueError(
            f"Multiple pixel_x_cm values found for {image_name}"
        )

    pixel_x_cm = float(match.iloc[0])

    print(f"Image: {image_name}")
    print(f"Pixel X: {pixel_x_cm:.4f} cm")
    px_per_m = pixel_x_cm*100
    pixel_length_m = px_per_m
    # hist = si.get_histogram(grains, px_per_m)
    summary = si.get_summary(grains, px_per_m)
    hist = seg.plot_histogram_of_axis_lengths(
        summary['major_axis_length']*1000,
        summary['minor_axis_length']*1000,
        binsize=0.25)


In [ ]:
def interactive_cleaning():
  global plot
  # You only need to run this cell if you want to interactively edit the segmentation results and/or if you want to draw a scale bar
  # If you want to draw a scale bar, you should change the 'scale_m' parameter before running this cell

  # Set up SAM 2.1 predictor for adding new grains
  predictor = SAM2ImagePredictor(sam)
  predictor.set_image(image)

  # Display interactive interface
  plot = si.GrainPlot(
    grains,
    image = image,
    predictor = predictor,
    blit = False, # blitting does not work in Colab
    color_palette = 'tab20b',
    figsize = (12, 8),  # inches
    scale_m = 0.1,      # meters; change this before launching 'GrainPlot'
    color_by = None,
    px_per_m = 1856.6   # px/m; alternative to 'scale_m'; will be overwritten if scale bar is drawn on image, using 'scale_m'
  )

  plot.activate()

plots

In [ ]:
def Plot_Auto_clean():
  global ROUNDNESS_THRESHOLD
  from skimage.measure import regionprops, regionprops_table
  import math
  from matplotlib import colormaps
  from matplotlib.patches import Polygon as MplPolygon
  import matplotlib.pyplot as plt
  ROUNDNESS_THRESHOLD = Roundness_Thresh.value
  def polygon_parts(geometry):
    """
    Return individual polygon parts from a Polygon or MultiPolygon geometry.
    """

    if geometry is None or geometry.is_empty:
        return []

    if geometry.geom_type == "Polygon":
        return [geometry]

    if geometry.geom_type == "MultiPolygon":
        return list(geometry.geoms)
    return []
  def add_polygon_to_axis(
    ax,
    geometry,
    face_colour,
    edge_colour="black",
    alpha=0.55,
    linewidth=0.8):
    """
    Add a Shapely Polygon or MultiPolygon to a Matplotlib axis.
    """

    for polygon in polygon_parts(geometry):

        coordinates = np.asarray(polygon.exterior.coords)

        patch = MplPolygon(
            coordinates,
            closed=True,
            facecolor=face_colour,
            edgecolor=edge_colour,
            linewidth=linewidth,
            alpha=alpha)

        ax.add_patch(patch)
  # -------------------------------------------------------------------------
  # Calculate mask roundness from the label raster
  # -------------------------------------------------------------------------
  initial_properties = regionprops_table(
        labels.astype(int),
        properties=(
            "label",
            "area",
            "perimeter"))

  initial_mask_data = pd.DataFrame(initial_properties)

  # Calculate roundness:
  # 1.0 = perfectly circular
  # lower values = increasingly irregular / elongated
  initial_mask_data["Roundness"] = np.where(
        initial_mask_data["perimeter"] > 0, (
            4
            * math.pi
            * initial_mask_data["area"]
            / (initial_mask_data["perimeter"] ** 2)),0,)

  # -------------------------------------------------------------------------
  # Split grains at the roundness threshold
  # -------------------------------------------------------------------------
  accepted_label_ids = initial_mask_data.loc[
        initial_mask_data["Roundness"] >= ROUNDNESS_THRESHOLD,
        "label"].astype(int).tolist()

  rejected_label_ids = initial_mask_data.loc[
        initial_mask_data["Roundness"] < ROUNDNESS_THRESHOLD,
        "label"].astype(int).tolist()

  # SegmentEveryGrain label IDs correspond to polygon-list positions + 1.
  # Extract the underlying Shapely polygon from each Grain object.
  accepted_grains = [
        grains[label_id - 1].polygon
        for label_id in accepted_label_ids
        if 0 < label_id <= len(grains)]

  rejected_grains = [
        grains[label_id - 1].polygon
        for label_id in rejected_label_ids
        if 0 < label_id <= len(grains)]

  print(f"Total masks: {len(initial_mask_data)}")
  print(f"Accepted masks (roundness >= {ROUNDNESS_THRESHOLD}): "
        f"{len(accepted_label_ids)}")
  print(f"Rejected masks (roundness < {ROUNDNESS_THRESHOLD}): "
        f"{len(rejected_label_ids)}")

  # -------------------------------------------------------------------------
  # Create accepted-only and rejected-only label rasters
  # -------------------------------------------------------------------------

  labels_filtered = np.where(
        np.isin(labels, accepted_label_ids),
        labels,
        0).astype(int)

  labels_rejected = np.where(
        np.isin(labels, rejected_label_ids),
        labels,
        0).astype(int)
  # =========================================================================
  # IMAGE 1
  # Accepted grains: roundness >= threshold
  # =========================================================================
  accepted_plot_output = (f"{fname}_roundness_accepted.png")
  fig, ax = plt.subplots(figsize=(15, 10))
  # Original image
  ax.imshow(image)
  paired_colours = colormaps["Paired"]
  # Draw accepted SegmentEveryGrain polygons
  for polygon_number, polygon in enumerate(accepted_grains):

        colour = paired_colours(
            polygon_number % paired_colours.N)
        add_polygon_to_axis(
            ax=ax,
            geometry=polygon,
            face_colour=colour,
            edge_colour="black",
            alpha=0.55,
            linewidth=0.8)
  # -------------------------------------------------------------------------
  # Major/minor axes and centroid for accepted grains
  # -------------------------------------------------------------------------

  accepted_regions = regionprops(labels_filtered)

  for region in accepted_regions:

        centroid_y, centroid_x = region.centroid
        orientation = region.orientation

        minor_x = (
            centroid_x
            + math.cos(orientation)
            * 0.5
            * region.minor_axis_length)

        minor_y = (
            centroid_y
            - math.sin(orientation)
            * 0.5
            * region.minor_axis_length)

        major_x = (
            centroid_x
            - math.sin(orientation)
            * 0.5
            * region.major_axis_length)
        major_y = (
            centroid_y
            - math.cos(orientation)
            * 0.5
            * region.major_axis_length)

        # Minor axis
        ax.plot(
            (centroid_x, minor_x),
            (centroid_y, minor_y),
            color="black",
            linewidth=1)

        # Major axis
        ax.plot(
            (centroid_x, major_x),
            (centroid_y, major_y),
            color="black",
            linewidth=1)

        # Centroid
        ax.plot(
            centroid_x,
            centroid_y,
            marker="o",
            markersize=2.5,
            color="black")

  ax.set_xlim(0, image.shape[1])
  ax.set_ylim(image.shape[0], 0)
  fig.tight_layout()
  fig.savefig(
        accepted_plot_output,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0,)

  plt.close(fig)
  print(
        f"Accepted-mask plot saved to: "
        f"{accepted_plot_output}")

  # =========================================================================
  # IMAGE 2
  # Rejected grains: roundness < threshold
  # =========================================================================

  rejected_plot_output = (f"{fname}_roundness_rejected.png")
  fig, ax = plt.subplots(figsize=(15, 10))
  # Original image
  ax.imshow(image)

  # Draw rejected polygons in red
  for polygon in rejected_grains:
        add_polygon_to_axis(
            ax=ax,
            geometry=polygon,
            face_colour="red",
            edge_colour="darkred",
            alpha=0.45,
            linewidth=1.0)

  # -------------------------------------------------------------------------
  # Centroid markers on rejected grains
  # -------------------------------------------------------------------------
  rejected_regions = regionprops(labels_rejected)
  for region in rejected_regions:
        centroid_y, centroid_x = region.centroid
        ax.plot(
            centroid_x,
            centroid_y,
            marker="o",
            markersize=2.5,
            color="darkred")
  ax.set_xlim(0, image.shape[1])
  ax.set_ylim(image.shape[0], 0)
  fig.tight_layout()
  fig.savefig(
        rejected_plot_output,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0)

  plt.close(fig)
  print(
        f"Rejected-mask plot saved to: "
        f"{rejected_plot_output}")
  from PIL import Image
  import matplotlib.pyplot as plt
  # -------------------------------------------------------------------------
  # Display accepted and rejected plots side by side in Google Colab
  # -------------------------------------------------------------------------

  accepted_image = Image.open(accepted_plot_output)
  rejected_image = Image.open(rejected_plot_output)
  fig, axes = plt.subplots(2, 1, figsize=(20, 14))
  axes[0].imshow(accepted_image)
  axes[0].set_title(
    f"Accepted grains (Roundness >= {ROUNDNESS_THRESHOLD})",
    fontsize=14)
  axes[0].axis("off")
  axes[1].imshow(rejected_image)
  axes[1].set_title(
    f"Removed grains (Roundness < {ROUNDNESS_THRESHOLD})",
    fontsize=14)
  axes[1].axis("off")

  plt.tight_layout()
  plt.show()

In [ ]:
def Extract_grain_data(
    labels,
    image,
    grains,
    fname,
    pixel_length_m,
    Roundness_Thresh,
):
    """
    Measure grains passing the roundness threshold, assign particle-size
    categories, and save:

    1. One row per accepted grain.
    2. One summary row per image.

    Parameters
    ----------
    labels : numpy.ndarray
        Label raster produced by SegmentEveryGrain.

    image : numpy.ndarray
        Original RGB image.

    grains : list
        SegmentEveryGrain polygon list or Grain-object list.

    fname : str
        Image filename or image identifier.

    pixel_length_m : float
        Ground-sampling distance in metres per pixel.

    Roundness_Thresh : float or ipywidgets.FloatSlider
        Minimum accepted roundness. A numeric value or widget can be passed.
    """

    import math
    import numpy as np
    import pandas as pd

    from pathlib import Path
    from scipy.stats import skew, kurtosis
    from skimage.measure import regionprops_table

    # ---------------------------------------------------------------------
    # Particle-size categories based on minor-axis length in millimetres
    # ---------------------------------------------------------------------

    categories = {
        "gravel": (2, 4),
        "pebbles": (4, 64),
        "cobbles": (64, 256),
        "boulders": (256, float("inf")),
    }

    # Accept either the slider widget or a numeric value.
    roundness_threshold = float(
        getattr(Roundness_Thresh, "value", Roundness_Thresh)
    )

    # ---------------------------------------------------------------------
    # Validate inputs
    # ---------------------------------------------------------------------

    labels = np.asarray(labels, dtype=np.int32)
    image = np.asarray(image)
    pixel_length_m = float(pixel_length_m)

    if labels.ndim != 2:
        raise ValueError(
            f"labels must be a 2D array. Received shape: {labels.shape}"
        )

    if image.shape[:2] != labels.shape:
        raise ValueError(
            "The image and label raster have different dimensions: "
            f"image={image.shape[:2]}, labels={labels.shape}"
        )

    if pixel_length_m <= 0:
        raise ValueError("pixel_length_m must be greater than zero.")

    if not 0.0 <= roundness_threshold <= 1.0:
        raise ValueError(
            "Roundness_Thresh must be between 0 and 1."
        )

    image_name = Path(fname).name
    image_stem = Path(fname).stem

    grain_csv_output = (
        f"{image_stem}_grain_measurements.csv"
    )

    summary_csv_output = (
        f"{image_stem}_grain_summary.csv"
    )

    total_pixels = labels.size

    # Area represented by one pixel, in square metres.
    pixel_area_m2 = pixel_length_m ** 2

    # ---------------------------------------------------------------------
    # Measure all labelled masks in pixels
    # ---------------------------------------------------------------------

    properties = regionprops_table(
        labels,
        intensity_image=image,
        properties=(
            "label",
            "area",
            "centroid",
            "major_axis_length",
            "minor_axis_length",
            "orientation",
            "perimeter",
            "max_intensity",
            "mean_intensity",
            "min_intensity",
        ),
    )

    all_mask_data = pd.DataFrame(properties)

    if all_mask_data.empty:
        print(f"No labelled masks were found for {image_name}.")
        return pd.DataFrame(), pd.DataFrame()

    # ---------------------------------------------------------------------
    # Calculate roundness in pixel units
    # ---------------------------------------------------------------------

    all_mask_data["Roundness_pixels"] = np.where(
        all_mask_data["perimeter"] > 0,
        (
            4.0
            * math.pi
            * all_mask_data["area"]
            / (all_mask_data["perimeter"] ** 2)
        ),
        0.0,
    )

    accepted_mask = (
        all_mask_data["Roundness_pixels"]
        >= roundness_threshold
    )

    accepted_label_ids = (
        all_mask_data.loc[accepted_mask, "label"]
        .astype(int)
        .tolist()
    )

    rejected_label_ids = (
        all_mask_data.loc[~accepted_mask, "label"]
        .astype(int)
        .tolist()
    )

    # ---------------------------------------------------------------------
    # Create an accepted-only label raster
    # ---------------------------------------------------------------------

    labels_filtered = np.where(
        np.isin(labels, accepted_label_ids),
        labels,
        0,
    ).astype(np.int32)

    # ---------------------------------------------------------------------
    # Measure accepted grains only
    # ---------------------------------------------------------------------

    accepted_properties = regionprops_table(
        labels_filtered,
        intensity_image=image,
        properties=(
            "label",
            "area",
            "centroid",
            "major_axis_length",
            "minor_axis_length",
            "orientation",
            "perimeter",
            "max_intensity",
            "mean_intensity",
            "min_intensity",
        ),
    )

    grain_data = pd.DataFrame(accepted_properties)

    if grain_data.empty:
        print(
            f"No masks passed the roundness threshold of "
            f"{roundness_threshold:.2f} for {image_name}."
        )

        return grain_data, pd.DataFrame()

    # ---------------------------------------------------------------------
    # Rename original pixel measurements
    # ---------------------------------------------------------------------

    grain_data = grain_data.rename(
        columns={
            "area": "area_pixels",
            "perimeter": "perimeter_pixels",
            "major_axis_length":
                "major_axis_length_pixels",
            "minor_axis_length":
                "minor_axis_length_pixels",
        }
    )

    # ---------------------------------------------------------------------
    # Convert measurements
    # ---------------------------------------------------------------------

    grain_data["area_m2"] = (
        grain_data["area_pixels"]
        * pixel_area_m2
    )

    grain_data["perimeter_m"] = (
        grain_data["perimeter_pixels"]
        * pixel_length_m
    )

    grain_data["major_axis_length_mm"] = (
        grain_data["major_axis_length_pixels"]
        * pixel_length_m
        * 1000.0
    )

    grain_data["minor_axis_length_mm"] = (
        grain_data["minor_axis_length_pixels"]
        * pixel_length_m
        * 1000.0
    )

    # ---------------------------------------------------------------------
    # Recalculate roundness using converted measurements
    # ---------------------------------------------------------------------

    grain_data["Roundness"] = np.where(
        grain_data["perimeter_m"] > 0,
        (
            4.0
            * math.pi
            * grain_data["area_m2"]
            / (grain_data["perimeter_m"] ** 2)
        ),
        0.0,
    )

    # Add the original pixel-based roundness by matching label IDs.
    roundness_lookup = (
        all_mask_data
        .set_index("label")["Roundness_pixels"]
    )

    grain_data["Roundness_pixels"] = (
        grain_data["label"]
        .map(roundness_lookup)
    )

    # The difference should be close to zero because roundness is
    # dimensionless.
    grain_data["roundness_difference"] = np.abs(
        grain_data["Roundness"]
        - grain_data["Roundness_pixels"]
    )

    # Apply the threshold again as a numerical consistency check.
    grain_data = grain_data.loc[
        grain_data["Roundness"]
        >= roundness_threshold
    ].copy()

    if grain_data.empty:
        print(
            "No masks remained after the roundness consistency "
            f"check for {image_name}."
        )

        return grain_data, pd.DataFrame()

    # ---------------------------------------------------------------------
    # Add image and polygon information
    # ---------------------------------------------------------------------

    grain_data.insert(
        0,
        "ImageName",
        image_name,
    )

    grain_data["polygon_list_index"] = (
        grain_data["label"].astype(int) - 1
    )

    grain_data["polygon_available"] = (
        grain_data["polygon_list_index"]
        .between(0, len(grains) - 1)
    )

    # ---------------------------------------------------------------------
    # Assign particle-size categories using minor-axis length
    # ---------------------------------------------------------------------

    grain_data["particle_size_category"] = "Unclassified"

    for category, bounds in categories.items():

        lower_bound, upper_bound = bounds

        category_mask = (
            grain_data["minor_axis_length_mm"]
            .gt(lower_bound)
            & grain_data["minor_axis_length_mm"]
            .le(upper_bound)
        )

        grain_data.loc[
            category_mask,
            "particle_size_category",
        ] = category

    # ---------------------------------------------------------------------
    # Calculate percentage of image covered by accepted grains
    # ---------------------------------------------------------------------

    accepted_pixel_area = grain_data["area_pixels"].sum()

    percentage_of_image = (
        100.0
        * accepted_pixel_area
        / total_pixels
    )

    # ---------------------------------------------------------------------
    # Summarise each particle-size category
    # ---------------------------------------------------------------------

    category_summary = {}

    for category in categories:

        category_data = grain_data.loc[
            grain_data["particle_size_category"]
            == category
        ].copy()

        count = len(category_data)

        areas = category_data["area_m2"]
        roundnesses = category_data["Roundness"]
        major_lengths = category_data[
            "major_axis_length_mm"
        ]
        minor_lengths = category_data[
            "minor_axis_length_mm"
        ]

        category_summary[category] = {
            "count":
                count,

            "mean_area_m2":
                areas.mean()
                if count else 0.0,

            "mean_roundness":
                roundnesses.mean()
                if count else 0.0,

            "mean_major_axis_length_mm":
                major_lengths.mean()
                if count else 0.0,

            "mean_minor_axis_length_mm":
                minor_lengths.mean()
                if count else 0.0,

            "range_area_m2":
                areas.max() - areas.min()
                if count else 0.0,

            "range_roundness":
                roundnesses.max() - roundnesses.min()
                if count else 0.0,

            "range_major_axis_length_mm":
                major_lengths.max() - major_lengths.min()
                if count else 0.0,

            "range_minor_axis_length_mm":
                minor_lengths.max() - minor_lengths.min()
                if count else 0.0,

            "min_area_m2":
                areas.min()
                if count else 0.0,

            "max_area_m2":
                areas.max()
                if count else 0.0,

            "min_roundness":
                roundnesses.min()
                if count else 0.0,

            "max_roundness":
                roundnesses.max()
                if count else 0.0,

            "min_major_axis_length_mm":
                major_lengths.min()
                if count else 0.0,

            "max_major_axis_length_mm":
                major_lengths.max()
                if count else 0.0,

            "min_minor_axis_length_mm":
                minor_lengths.min()
                if count else 0.0,

            "max_minor_axis_length_mm":
                minor_lengths.max()
                if count else 0.0,

            "total_area_m2":
                areas.sum()
                if count else 0.0,

            "variance_area_m2":
                areas.var()
                if count > 1 else 0.0,

            "variance_roundness":
                roundnesses.var()
                if count > 1 else 0.0,

            "std_dev_area_m2":
                areas.std()
                if count > 1 else 0.0,

            "std_dev_roundness":
                roundnesses.std()
                if count > 1 else 0.0,

            "skewness_area":
                skew(
                    areas.to_numpy(),
                    bias=False,
                )
                if count > 2 else 0.0,

            "skewness_roundness":
                skew(
                    roundnesses.to_numpy(),
                    bias=False,
                )
                if count > 2 else 0.0,

            "kurtosis_area":
                kurtosis(
                    areas.to_numpy(),
                    bias=False,
                )
                if count > 3 else 0.0,

            "kurtosis_roundness":
                kurtosis(
                    roundnesses.to_numpy(),
                    bias=False,
                )
                if count > 3 else 0.0,
        }

    # ---------------------------------------------------------------------
    # Flatten category statistics into one row
    # ---------------------------------------------------------------------

    summary_data = {}

    for category, category_values in category_summary.items():

        for statistic, value in category_values.items():

            column_name = f"{category}_{statistic}"

            summary_data[column_name] = value

    # Include grains below the lowest category boundary.
    unclassified_count = (
        grain_data["particle_size_category"]
        .eq("Unclassified")
        .sum()
    )

    summary_data.update(
        {
            "total_masks_before_filter":
                len(all_mask_data),

            "accepted_mask_count":
                len(grain_data),

            "rejected_mask_count":
                len(rejected_label_ids),

            "unclassified_count":
                int(unclassified_count),

            "roundness_threshold":
                roundness_threshold,

            "percentageOfImage":
                percentage_of_image,

            "pixel_length_m":
                pixel_length_m,

            "pixel_area_m2":
                pixel_area_m2,

            "ImageName":
                image_name,
        }
    )

    stats = pd.DataFrame([summary_data])

    # ---------------------------------------------------------------------
    # Save outputs
    # ---------------------------------------------------------------------

    grain_data.to_csv(
        grain_csv_output,
        index=False,
    )

    stats.to_csv(
        summary_csv_output,
        index=False,
    )

    # ---------------------------------------------------------------------
    # Report results
    # ---------------------------------------------------------------------

    print(f"Image: {image_name}")

    print(
        f"Total masks before filtering: "
        f"{len(all_mask_data)}"
    )

    print(
        f"Accepted masks: {len(grain_data)} "
        f"(roundness >= {roundness_threshold:.2f})"
    )

    print(
        f"Rejected masks: {len(rejected_label_ids)} "
        f"(roundness < {roundness_threshold:.2f})"
    )

    print(
        f"Unclassified accepted grains: "
        f"{unclassified_count}"
    )

    print(
        f"Accepted masks cover "
        f"{percentage_of_image:.2f}% of the image."
    )

    print(f"Image summary saved to: "
        f"{summary_csv_output}")

    return stats